### Fraud Detection Models - Practice
### For ELK lab.

In [9]:
# ============================================================
# Simple XGBoost Fraud Detection Model
# Purpose: Generate metrics for ELK monitoring
# ============================================================

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score, confusion_matrix
)
import xgboost as xgb
from datetime import datetime


import pandas as pd

# Load the dataset
print("Loading credit card fraud dataset...")
df = pd.read_csv('../dataset/creditcard.csv')

print(f"Dataset loaded: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Fraud count: {df['Class'].sum()}")



print("="*70)
print("XGBOOST FRAUD DETECTION MODEL".center(70))
print("="*70)

# ---------------------------------------------------------
# 1. PREPARE DATA
# ---------------------------------------------------------
print("\nStep 1: Preparing data...")

# Separate features and target
# Time, Amount, Class, V1-V28 -- all the columns.
X = df.drop('Class', axis=1).values
y = df['Class'].values

# Split data (stratified to preserve fraud ratio)
# Keep same fraud ratio in train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  
)

print(f"   Training set: {len(X_train):,} samples ({y_train.sum()} frauds)")
print(f"   Test set: {len(X_test):,} samples ({y_test.sum()} frauds)")


Loading credit card fraud dataset...
Dataset loaded: (284807, 31)
Columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
Fraud count: 492
                    XGBOOST FRAUD DETECTION MODEL                     

Step 1: Preparing data...
   Training set: 227,845 samples (394 frauds)
   Test set: 56,962 samples (98 frauds)


In [10]:
# Test 1: Is df loaded?
print(f"Dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage().sum() / 1024**2:.1f} MB")

# Test 2: Can we split data?
X_test = df.drop('Class', axis=1).values
y_test = df['Class'].values
print(f"Data split works: X shape = {X_test.shape}")

# Test 3: Can we import XGBoost?
import xgboost as xgb
print(f"XGBoost version: {xgb.__version__}")

# Test 4: Quick mini-train (should take <1 second)
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X_test[:1000], y_test[:1000], test_size=0.2)
quick_model = xgb.XGBClassifier(n_estimators=10, random_state=42)
quick_model.fit(X_tr, y_tr)
print(f"Quick training works!")


Dataset shape: (284807, 31)
Memory usage: 67.4 MB
Data split works: X shape = (284807, 30)
XGBoost version: 3.1.1
Quick training works!


In [21]:

# ---------------------------------------------------------
# 2. CALCULATE SCALE_POS_WEIGHT (Handle Imbalance)

# Ideas: sample specific? or even cost base, some adaptive weigting.
# similar to custom anatomy loss i tried in old project. 
# focal loss approach on hard to classify problems - Find which frauds are hardest to detect, retrain.
# ---------------------------------------------------------
from numpy import sqrt


print("\nStep 2: Handling class imbalance...")


# ## Attempt 1:
# scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
# print(f"   scale_pos_weight = {scale_pos_weight:.1f}")
# print(f"   → This tells XGBoost: 'Fraud is {scale_pos_weight:.0f}x more important!'")


# ## Attempt 2:0
# weighted scaling turned bad for cost-based item. number came - 2854.1 (Cost Ratio is Too Extreme)

# ## Attempt 3:
standard_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

avg_fraud_amount = df[df['Class']==1]['Amount'].mean()
avg_normal_amount = df[df['Class']==0]['Amount'].mean()

cost_missing_fraud = avg_fraud_amount  # $122
cost_false_alarm = 25  

cost_ratio = cost_missing_fraud / cost_false_alarm

adjustment_factor = min(sqrt(cost_ratio), 3.0)  # Cap at 3x
adaptive_weight = standard_weight * adjustment_factor

print(f"\nWeighting Analysis:")
print(f"  Standard weight (count-based): {standard_weight:.1f}")
print(f"  Average fraud amount: ${avg_fraud_amount:.2f}")
print(f"  Cost of missing fraud: ${cost_missing_fraud:.2f}")
print(f"  Cost of false alarm: ${cost_false_alarm:.2f}")
print(f"  Cost ratio: {cost_ratio:.2f}")
print(f"  Adjustment factor (capped): {adjustment_factor:.2f}x")
print(f"  Adaptive weight: {adaptive_weight:.1f}")
print(f"\n  Using: {adaptive_weight:.1f} (moderate cost adjustment)")


Step 2: Handling class imbalance...

Weighting Analysis:
  Standard weight (count-based): 577.3
  Average fraud amount: $122.21
  Cost of missing fraud: $122.21
  Cost of false alarm: $25.00
  Cost ratio: 4.89
  Adjustment factor (capped): 2.21x
  Adaptive weight: 1276.4

  Using: 1276.4 (moderate cost adjustment)


In [22]:

"""
- F_beta = (1 + beta²) × (precision × recall) / (beta² × precision + recall)
- F1 = 2 × (precision × recall) / (precision + recall)
    - precision and recall are balanced equally
- F2 = 5 × (precision × recall) / (4 × precision + recall)

## XGBoost items:
min_child_weight - min samples to create leaf
subsample + colsample_bytree - randomness
lets add gamma too.
"""


'\n- F_beta = (1 + beta²) × (precision × recall) / (beta² × precision + recall)\n- F1 = 2 × (precision × recall) / (precision + recall)\n    - precision and recall are balanced equally\n- F2 = 5 × (precision × recall) / (4 × precision + recall)\n\n## XGBoost items:\nmin_child_weight - min samples to create leaf\nsubsample + colsample_bytree - randomness\nlets add gamma too.\n'

In [23]:
from sklearn.metrics import fbeta_score, precision_recall_curve
import json

# ---------------------------------------------------------
# 3. TRAIN XGBOOST MODEL
# ---------------------------------------------------------
print("\nStep 3: Training XGBoost model...")
print(f"  Using adaptive scale_pos_weight: {adaptive_weight:.1f}")

# XGBoost parameters
params = {
    'max_depth': 5,
    'learning_rate': 0.05,
    'n_estimators': 100,
    'scale_pos_weight': adaptive_weight,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'random_state': 42,
    'use_label_encoder': False
}

# Train model
start_time = datetime.now()
model = xgb.XGBClassifier(**params)
model.fit(X_train, y_train, verbose=False)
training_time = (datetime.now() - start_time).total_seconds()

print(f"  Model trained in {training_time:.2f} seconds")

# ---------------------------------------------------------
# GET PREDICTIONS AND PROBABILITIES
# ---------------------------------------------------------
print("\nStep 4: Generating predictions...")

y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred_default = model.predict(X_test)

print("  Probabilities and predictions generated")

# ---------------------------------------------------------
# EXPERIMENT 1: THRESHOLD TUNING
# ---------------------------------------------------------
print("\n" + "="*70)
print("EXPERIMENT 1: THRESHOLD ANALYSIS".center(70))
print("="*70)

# Test more thresholds, especially lower ones for higher recall
thresholds_to_test = [0.2, 0.3, 0.4, 0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.82, 0.85, 0.875, 0.9, 0.95, 0.97]
results = []

print("\nTesting different decision thresholds:")
print("-"*70)
print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<12} {'F2':<12}")
print("-"*70)

for threshold in thresholds_to_test:
    y_pred_thresh = (y_pred_proba >= threshold).astype(int)
    
    precision = precision_score(y_test, y_pred_thresh, zero_division=0)
    recall = recall_score(y_test, y_pred_thresh, zero_division=0)
    f1 = f1_score(y_test, y_pred_thresh, zero_division=0)
    f2 = fbeta_score(y_test, y_pred_thresh, beta=2, zero_division=0)
    
    results.append({
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'f2': f2,
        'y_pred': y_pred_thresh
    })
    
    print(f"{threshold:<12.2f} {precision:<12.3f} {recall:<12.3f} {f1:<12.3f} {f2:<12.3f}")

print("-"*70)

# ---------------------------------------------------------
# SMART THRESHOLD SELECTION: Prioritize Recall!
# ---------------------------------------------------------
print("\nSMART THRESHOLD SELECTION (Fraud Detection Priority):")
print("-"*70)

# Strategy: Find threshold that maximizes F2 while keeping recall >= 95%
MIN_RECALL = 0.95
valid_results = [r for r in results if r['recall'] >= MIN_RECALL]

if valid_results:
    # Among thresholds with recall >= 95%, pick the one with best F2
    best_valid_idx = max(range(len(valid_results)), key=lambda i: valid_results[i]['f2'])
    best_valid = valid_results[best_valid_idx]
    
    # Find this in original results list
    optimal_idx = next(i for i, r in enumerate(results) if r['threshold'] == best_valid['threshold'])
    
    print(f"  Strategy: Maximize F2 with Recall >= {MIN_RECALL*100:.0f}%")
    print(f"  Selected Threshold: {best_valid['threshold']:.2f}")
    print(f"    Precision: {best_valid['precision']:.4f}")
    print(f"    Recall:    {best_valid['recall']:.4f} (>= {MIN_RECALL*100:.0f}%)")
    print(f"    F2 Score:  {best_valid['f2']:.4f} (optimized)")
else:
    # Fallback: No threshold gives 95% recall, so just maximize F2
    optimal_idx = max(range(len(results)), key=lambda i: results[i]['f2'])
    print(f"  Strategy: Maximize F2 (no threshold achieved {MIN_RECALL*100:.0f}% recall)")
    print(f"  Selected Threshold: {results[optimal_idx]['threshold']:.2f}")
    print(f"    Best achievable recall: {results[optimal_idx]['recall']:.4f}")

# Also show traditional F1 optimal for comparison
optimal_f1_idx = max(range(len(results)), key=lambda i: results[i]['f1'])
optimal_f2_idx = max(range(len(results)), key=lambda i: results[i]['f2'])

print(f"\nFor comparison:")
print(f"  F1-optimal threshold: {results[optimal_f1_idx]['threshold']:.2f} → F1={results[optimal_f1_idx]['f1']:.4f}, Recall={results[optimal_f1_idx]['recall']:.4f}")
print(f"  F2-optimal threshold: {results[optimal_f2_idx]['threshold']:.2f} → F2={results[optimal_f2_idx]['f2']:.4f}, Recall={results[optimal_f2_idx]['recall']:.4f}")
print(f"  Our choice (F2 + recall constraint): {results[optimal_idx]['threshold']:.2f}")

# Use our smart selection
optimal_threshold = results[optimal_idx]['threshold']
y_pred_optimal = results[optimal_idx]['y_pred']
precision_opt = results[optimal_idx]['precision']
recall_opt = results[optimal_idx]['recall']
f1_opt = results[optimal_idx]['f1']
f2_opt = results[optimal_idx]['f2']

# ---------------------------------------------------------
# EXPERIMENT 2: COMPARISON WITH DEFAULT
# ---------------------------------------------------------
print("\n" + "="*70)
print("COMPARISON: Default (0.5) vs Optimal Threshold".center(70))
print("="*70)

# Default threshold performance
y_pred_05 = (y_pred_proba >= 0.5).astype(int)
precision_05 = precision_score(y_test, y_pred_05, zero_division=0)
recall_05 = recall_score(y_test, y_pred_05, zero_division=0)
f1_05 = f1_score(y_test, y_pred_05, zero_division=0)
f2_05 = fbeta_score(y_test, y_pred_05, beta=2, zero_division=0)

print(f"\nDefault Threshold (0.50):")
print(f"  Precision: {precision_05:.4f}")
print(f"  Recall:    {recall_05:.4f}")
print(f"  F1 Score:  {f1_05:.4f}")
print(f"  F2 Score:  {f2_05:.4f}")

print(f"\nOptimal Threshold ({optimal_threshold:.2f}) - Fraud Detection Focus:")
print(f"  Precision: {precision_opt:.4f}  (change: {precision_opt - precision_05:+.4f})")
print(f"  Recall:    {recall_opt:.4f}  (change: {recall_opt - recall_05:+.4f})")
print(f"  F1 Score:  {f1_opt:.4f}  (change: {f1_opt - f1_05:+.4f})")
print(f"  F2 Score:  {f2_opt:.4f}  (change: {f2_opt - f2_05:+.4f})")

# ---------------------------------------------------------
# EXPERIMENT 3: F-BETA SCORES
# ---------------------------------------------------------
print("\n" + "="*70)
print("EXPERIMENT 3: F-BETA SCORE ANALYSIS".center(70))
print("="*70)

# Calculate multiple F-beta scores at optimal threshold
f05 = fbeta_score(y_test, y_pred_optimal, beta=0.5)
f10 = f1_opt
f15 = fbeta_score(y_test, y_pred_optimal, beta=1.5)
f20 = f2_opt

print(f"\nF-beta scores at optimal threshold ({optimal_threshold:.2f}):")
print("-"*70)
print(f"{'Beta':<10} {'Score':<12} {'Interpretation':<50}")
print("-"*70)
print(f"{'0.5':<10} {f05:<12.4f} {'Precision-weighted (2x precision importance)':<50}")
print(f"{'1.0':<10} {f10:<12.4f} {'Balanced (equal importance)':<50}")
print(f"{'1.5':<10} {f15:<12.4f} {'Slight recall preference':<50}")
print(f"{'2.0':<10} {f20:<12.4f} {'Recall-weighted (2x recall importance)':<50}")

print(f"\nFraud Detection Assessment:")
if recall_opt >= 0.95 and f20 >= 0.80:
    print(f"  EXCELLENT: Recall={recall_opt:.1%}, F2={f20:.4f}")
    print(f"  Catching >95% of frauds with strong overall performance")
elif recall_opt >= 0.90 and f20 >= 0.75:
    print(f"  GOOD: Recall={recall_opt:.1%}, F2={f20:.4f}")
    print(f"  Strong fraud detection with acceptable false alarms")
elif recall_opt >= 0.85:
    print(f"  ACCEPTABLE: Recall={recall_opt:.1%}, F2={f20:.4f}")
    print(f"  Reasonable fraud detection")
else:
    print(f"  NEEDS IMPROVEMENT: Recall={recall_opt:.1%}, F2={f20:.4f}")
    print(f"  Missing too many frauds")

# ---------------------------------------------------------
# FINAL DETAILED EVALUATION
# ---------------------------------------------------------
print("\n" + "="*70)
print("FINAL MODEL PERFORMANCE (Optimal Threshold)".center(70))
print("="*70)

# All metrics with optimal threshold
accuracy_opt = accuracy_score(y_test, y_pred_optimal)
auc_opt = roc_auc_score(y_test, y_pred_proba)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_optimal).ravel()
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

print(f"\nDecision Threshold: {optimal_threshold:.2f} (F2-optimized with recall constraint)")
print(f"Training Time: {training_time:.2f} seconds")

print(f"\nCore Metrics:")
print(f"  Accuracy:  {accuracy_opt:.4f}  (Overall correctness)")
print(f"  Precision: {precision_opt:.4f}  (When predicting fraud, correct {precision_opt*100:.1f}% of time)")
print(f"  Recall:    {recall_opt:.4f}  (Catching {recall_opt*100:.1f}% of all frauds) *** PRIMARY GOAL ***")
print(f"  F1 Score:  {f1_opt:.4f}  (Harmonic mean of precision & recall)")
print(f"  F2 Score:  {f2_opt:.4f}  (Recall-weighted for fraud detection) *** OPTIMIZED ***")
print(f"  AUC-ROC:   {auc_opt:.4f}  (Discriminative ability)")

print(f"\nError Analysis:")
print(f"  False Positive Rate: {fpr:.4f}  ({fp:,} normal transactions flagged)")
print(f"  False Negative Rate: {fnr:.4f}  ({fn:,} frauds missed) *** MINIMIZED ***")
print(f"  Specificity:         {1-fpr:.4f}  (Correctly identifying normal)")
print(f"  Sensitivity:         {recall_opt:.4f}  (Correctly identifying fraud)")

print(f"\nConfusion Matrix:")
print(f"                    Predicted")
print(f"                 Normal    Fraud")
print(f"  Actual Normal  {tn:>7,}  {fp:>7,}")
print(f"  Actual Fraud   {fn:>7,}  {tp:>7,}")

print(f"\nBusiness Impact:")
total_frauds = tp + fn
caught_frauds = tp
missed_frauds = fn
false_alarms = fp

print(f"  Total frauds in test set:  {total_frauds:,}")
print(f"  Frauds caught:              {caught_frauds:,} ({caught_frauds/total_frauds*100:.1f}%)")
print(f"  Frauds missed:              {missed_frauds:,} ({missed_frauds/total_frauds*100:.1f}%)")
print(f"  False alarms:               {false_alarms:,}")
if caught_frauds > 0:
    print(f"  Fraud-to-alarm ratio:       1:{false_alarms/caught_frauds:.1f}")

avg_fraud_value = 122
savings = caught_frauds * avg_fraud_value
false_alarm_cost = false_alarms * 25
net_value = savings - false_alarm_cost

print(f"\nFinancial Impact (estimated):")
print(f"  Fraud prevented:    ${savings:,} ({caught_frauds} × ${avg_fraud_value})")
print(f"  False alarm cost:   ${false_alarm_cost:,} ({false_alarms} × $25)")
print(f"  Net value:          ${net_value:,}")
print(f"  ROI:                {(net_value/false_alarm_cost)*100:.0f}%")

# ---------------------------------------------------------
# METRICS FOR ELK LOGGING
# ---------------------------------------------------------
print("\n" + "="*70)
print("METRICS TO LOG FOR ELK MONITORING".center(70))
print("="*70)

metrics_to_log = {
    'timestamp': datetime.now().isoformat(),
    'iteration': 1,
    'model': 'XGBoost',
    'optimization_strategy': 'F2_with_recall_constraint',
    
    # Core metrics
    'accuracy': round(accuracy_opt, 4),
    'precision': round(precision_opt, 4),
    'recall': round(recall_opt, 4),
    'f1_score': round(f1_opt, 4),
    'f2_score': round(f2_opt, 4),
    'auc_roc': round(auc_opt, 4),
    
    # Confusion matrix
    'true_positive': int(tp),
    'true_negative': int(tn),
    'false_positive': int(fp),
    'false_negative': int(fn),
    
    # Error rates
    'false_positive_rate': round(fpr, 4),
    'false_negative_rate': round(fnr, 4),
    'specificity': round(1-fpr, 4),
    'sensitivity': round(recall_opt, 4),
    
    # Business metrics
    'frauds_caught': int(caught_frauds),
    'frauds_missed': int(missed_frauds),
    'false_alarms': int(false_alarms),
    'estimated_savings': int(savings),
    'estimated_cost': int(false_alarm_cost),
    'net_value': int(net_value),
    
    # Model config
    'decision_threshold': round(optimal_threshold, 2),
    'scale_pos_weight': round(adaptive_weight, 1),
    'max_depth': params['max_depth'],
    'learning_rate': params['learning_rate'],
    'n_estimators': params['n_estimators'],
    'training_time_seconds': round(training_time, 2)
}

print("\nJSON log entry for Logstash:")
print("-"*70)
print(json.dumps(metrics_to_log, indent=2))

print("\n" + "="*70)
print("MODEL READY FOR ELK INTEGRATION!".center(70))
print("="*70)


Step 3: Training XGBoost model...
  Using adaptive scale_pos_weight: 1276.4


d:\JoelDesktop folds_24\NEU FALL2025\MLops IE7374 18008\MasterRepo & LabRepo\mlops-labs-portfolio\ELK_Stack_Lab\elk_venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [14:51:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  Model trained in 0.95 seconds

Step 4: Generating predictions...
  Probabilities and predictions generated

                   EXPERIMENT 1: THRESHOLD ANALYSIS                   

Testing different decision thresholds:
----------------------------------------------------------------------
Threshold    Precision    Recall       F1           F2          
----------------------------------------------------------------------
0.20         0.055        0.980        0.105        0.226       
0.30         0.091        0.978        0.166        0.331       
0.40         0.133        0.978        0.235        0.431       
0.50         0.194        0.976        0.324        0.541       
0.60         0.278        0.970        0.432        0.647       
0.65         0.336        0.970        0.499        0.704       
0.70         0.404        0.967        0.570        0.757       
0.75         0.506        0.967        0.664        0.818       
0.80         0.580        0.965        0.725        

## First time analysis
- True frauds caught:       479  (31%)
- False alarms:             1,069  (69%)
- Recall = 97.4%** - catching 479 out of 492 frauds. Only missing 13 frauds (2.6%).
- Precision = 30.9%** - **high false alarm rate**



In [ ]:
"""
scale_pos_weight - N

Weighted Loss = Loss(normal_1) + ... + Loss(normal_N) + N × Loss(fraud_1)
              ≈ N × normal_loss + N × fraud_loss

both classes contribute equally to the loss

gradient_fraud = (prediction - actual_label) × N

scale_pos_weight = count_negative_class / count_positive_class
usually, set to ratio of normal to fraud samples

---------------------------------------------------------------------------------------------------
XGBoost:
    Each new tree focuses on fixing the remaining errors
    Tree 3: Fixes errors from (Tree 1 + Tree 2)
    Tree 4: Fixes errors from (Tree 1 + Tree 2 + Tree 3)
    Final Prediction = Sum of predictions from all trees

= Loss(predictions, actual) + Ω(model complexity)
- Ω penalizes: deep trees, many leaves, large weights


---------------------------------------------------------------------------------------------------

# Current predictions
predictions_current = Tree_1 + Tree_2 + ... + Tree_k

# How wrong are we?
error = actual_labels - predictions_current

# Gradient: direction to reduce error
gradient = ∂Loss/∂predictions  

# Build Tree_k+1 to predict the gradient
Tree_k+1 = train_tree(features, target=gradient)

# Update predictions
predictions_new = predictions_current + learning_rate × Tree_k+1


- Each tree is a specialist.
prediction score = 0.3 × Tree_1(x) + 0.4 × Tree_2(x) + ... + 0.1 × Tree_100(x)

---------------------------------------------------------------------------------------------------

## objective = 'binary:logistic'

'binary:logistic':  # Your choice
→ Output: Probability (0.0 to 1.0)
→ Loss: Logistic loss (log loss)

'binary:hinge':
→ Output: Classification (-1 or +1)
→ Loss: Hinge loss (SVM-style)

'binary:logitraw':
→ Output: Raw logit scores (unbounded)

         1
p = ──────────────
    1 + e^(-score)
Converts any score (-∞ to +∞) into probability (0 to 1)


## eval_metric = 'auc'. Area Under ROC Curve. 

    **ROC Curve plots:**
    - X-axis: False Positive Rate (false alarms)
    - Y-axis: True Positive Rate (caught frauds)


## model.fit(X_train, y_train, verbose=False)

verbose = True
[20] train-auc:0.9456  valid-auc:0.9187
[30] train-auc:0.9587  valid-auc:0.9276
...
[100] train-auc:0.9789 valid-auc:0.9534

Shows AUC after each tree is added.

## KEY CONCEPTS SUMMARY:

scale_pos_weight = Makes minority class errors matter more (balances seesaw)
XGBoost = Ensemble of trees that iteratively fix each other's mistakes
max_depth = Tree complexity (5 = good spot, can debug later)
learning_rate = Step size (0.05 = conservative, needs more trees)
n_estimators = Number of trees (100 = standard starting point)
AUC = Threshold-independent performance measure (0.95 = excellent)


"""